# 8.3 — Closed-Form Regression

**Chapter 8, section 8.2.2**, and the code behind **Exercise 4**.

**The question this notebook answers:** why does the exact solution to a linear regression need
only one pass over the data, however many rows there are — and what stops it working anyway?

The normal equations give the coefficients directly, with no learning rate and no iteration:

$$\theta = (X^T X)^{-1} X^T y.$$

The two quantities on the right are the whole reason this distributes. $X^T X$ is $d \times d$ and
$X^T y$ has length $d$, **whatever $n$ is** — a hundred rows or a hundred billion produce a matrix
of exactly the same size. Both are sums of independent per-example contributions, so both are
computed the way every other aggregate in this chapter is computed, and both are small enough to
come back to the driver. What is left is a $d \times d$ linear system, solved locally.

The limit is therefore entirely a function of $d$, and the last section measures it.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-8.3")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The data

A regression problem small enough to check by hand: three features, an intercept folded in as a
constant column, and coefficients known in advance so the recovered ones can be compared against
them.

In [2]:
np.random.seed(777)

size = 20000
d = 3
theta_true = np.array([5.0, -1.0, 1.0, 2.0])   # intercept first, then the three coefficients

feats = np.random.rand(size, d)
noise = np.random.rand(size) * 0.1
Xlocal = np.hstack([np.ones((size, 1)), feats])       # the constant column is the intercept
ylocal = Xlocal @ theta_true + noise

rows = sc.parallelize(list(zip(ylocal, Xlocal)), 4)   # (label, feature vector) as everywhere else
rows.cache()
print("rows:", rows.count(), " parameters:", Xlocal.shape[1])
rows.take(2)

rows: 20000  parameters: 4


[(np.float64(5.333570668303061),
  array([1.        , 0.15266373, 0.30235661, 0.06203641])),
 (np.float64(7.259768261153993),
  array([1.        , 0.45986034, 0.83525338, 0.92699705]))]

---

## 2. $X^T X$ from a `RowMatrix`

`RowMatrix` is Spark's distributed matrix stored as an RDD of local row vectors — the natural
representation when a matrix has many rows and few columns. `computeGramianMatrix()` computes
$X^T X$ across the partitions and returns it as an ordinary local matrix.

The size of what comes back is the point: it is $d \times d$, and it would be $d \times d$ if the
RDD held a billion rows.

> **A message you will probably see, and should ignore.** The cell below may print a
> `BrokenPipeError` traceback from a Spark Python worker before it prints its results. It is an
> artifact of PySpark 4.2: handing a Python RDD to `pyspark.mllib` converts it to a JVM RDD, and
> the JVM stops reading before the worker has finished flushing. The worker writes to the JVM's
> standard error, so it cannot be silenced from this process, and the computation is unaffected —
> the checks two sections below confirm the Gramian is exactly right. It does not appear anywhere
> else in this chapter, because this is the only notebook that uses `pyspark.mllib`.

In [3]:
from pyspark.mllib.linalg.distributed import RowMatrix

feature_rows = rows.map(lambda r: r[1])
mat = RowMatrix(feature_rows)

gramian = mat.computeGramianMatrix().toArray()
print("rows in the distributed matrix:", mat.numRows())
print("shape of what came back to the driver:", gramian.shape)
print(gramian.round(2))

Traceback (most recent call last):
  File "/Users/dt/.venvs/cs777-spark/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 262, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dt/.venvs/cs777-spark/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


rows in the distributed matrix: 20000
shape of what came back to the driver: (4, 4)
[[20000.    9977.6   9954.85 10006.32]
 [ 9977.6   6647.05  4962.83  4996.17]
 [ 9954.85  4962.83  6619.28  4974.44]
 [10006.32  4996.17  4974.44  6674.43]]


---

## 3. $X^T y$ with one `treeAggregate`

The $j$-th entry of $X^T y$ is $\sum_i x_{ij} y_i$ — a sum of per-example contributions, exactly
like a gradient. It uses the same accumulator pattern: a zero vector, a `seqOp` that folds in one
example, and a `combOp` that adds two accumulators componentwise.

In [4]:
d_input = len(rows.first()[1])

Xty = rows.treeAggregate(
    np.zeros(d_input),
    lambda acc, r: acc + r[1] * r[0],        # seqOp: one example's contribution
    lambda a, b: a + b,                      # combOp: componentwise addition
)
print("X^T y:", Xty.round(2), " length:", len(Xty))

X^T y: [120989.67  58696.41  61876.07  63859.37]  length: 4


---

## 4. The solve, on the driver

`np.linalg.solve` solves the system $(X^T X)\theta = X^T y$ without forming the inverse, which is
both faster and numerically better behaved than computing $(X^T X)^{-1}$ and multiplying.

This is the only step that is not distributed, and it is the step whose cost is $O(d^3)$.

In [5]:
theta_closed = np.linalg.solve(gramian, Xty)

print("recovered: ", theta_closed.round(6))
print("generated: ", theta_true)
print("\nThe intercept is above 5 by about half the noise range, which is what an additive")
print("uniform error on [0, 0.1] with mean 0.05 should do to it.")

recovered:  [ 5.049761 -0.999106  0.999309  2.00025 ]
generated:  [ 5. -1.  1.  2.]

The intercept is above 5 by about half the noise range, which is what an additive
uniform error on [0, 0.1] with mean 0.05 should do to it.


### Checking it against two other routes

The first check is arithmetic: the same normal equations solved locally on the whole matrix. This
is only possible because the dataset is small enough to collect, which is exactly the situation
the distributed version exists to escape — it is a check, not a method.

In [6]:
theta_local = np.linalg.solve(Xlocal.T @ Xlocal, Xlocal.T @ ylocal)
assert np.allclose(theta_closed, theta_local), "the distributed and local normal equations disagree"
print("distributed normal equations == local normal equations")

# And against the Gramian formed locally, entry by entry, so the aggregation is checked too.
assert np.allclose(gramian, Xlocal.T @ Xlocal)
assert np.allclose(Xty, Xlocal.T @ ylocal)
print("computeGramianMatrix() == X^T X   and   treeAggregate == X^T y")

distributed normal equations == local normal equations
computeGramianMatrix() == X^T X   and   treeAggregate == X^T y


The second check is Spark's own. `LinearRegression(solver="normal")` tells the estimator to use
the weighted-least-squares route rather than L-BFGS, which is the same computation this notebook
has just performed by hand. The estimator manages its own intercept, so the constant column is
dropped before handing it the features.

In [7]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.linalg import Vectors

df = rows.map(lambda r: (float(r[0]), Vectors.dense(r[1][1:]))).toDF(["label", "features"])
df.cache(); df.count()

lr = LinearRegression(solver="normal", regParam=0.0, standardization=False)
model = lr.fit(df)

print("MLlib solver='normal':", np.round([model.intercept] + list(model.coefficients), 6))
print("by hand             :", theta_closed.round(6))

assert np.allclose([model.intercept] + list(model.coefficients), theta_closed, atol=1e-6)
print("\nThey agree to 1e-6.")

MLlib solver='normal': [ 5.049761 -0.999106  0.999309  2.00025 ]
by hand             : [ 5.049761 -0.999106  0.999309  2.00025 ]

They agree to 1e-6.


### And against gradient descent

The other route to the same minimum, for comparison. It needs a learning rate, it needs a stopping
rule, and it needs one pass over the data per iteration — and it arrives at the same place.

In [8]:
learningRate = 1
num_iteration = 30
theta = np.zeros(d_input)

for i in range(num_iteration):
    gradientAndCost = rows.map(lambda x: (x[1], (x[0] - np.dot(x[1], theta))))\
                          .map(lambda x: (x[0] * x[1], x[1]**2))\
                          .reduce(lambda x, y: (x[0] + y[0], x[1] + y[1]))
    cost = gradientAndCost[1] / (2 * size)
    gradient = (-1 / float(size)) * gradientAndCost[0]
    theta = theta - learningRate * gradient
    if i % 5 == 0 or i == num_iteration - 1:
        print(f"{i:3} Cost={cost:.8f} theta={theta.round(5)}")

print("\nclosed form      :", theta_closed.round(5))
print("gradient descent :", theta.round(5))
print(f"largest coefficient difference after {num_iteration} iterations: "
      f"{np.abs(theta - theta_closed).max():.2e}")

  0 Cost=18.54773941 theta=[6.04948 2.93482 3.0938  3.19297]
  5 Cost=1.76139646 theta=[2.99793 0.39731 1.21096 1.61898]


 10 Cost=0.23040779 theta=[4.27256 0.3999  1.63228 2.24996]
 15 Cost=0.06244650 theta=[ 4.19038 -0.11723  1.38718  2.13823]


 20 Cost=0.02957053 theta=[ 4.44617 -0.32485  1.35493  2.19341]
 25 Cost=0.01672470 theta=[ 4.55292 -0.51728  1.27589  2.17093]


 29 Cost=0.01103043 theta=[ 4.64203 -0.62068  1.23346  2.15919]

closed form      : [ 5.04976 -0.99911  0.99931  2.00025]
gradient descent : [ 4.64203 -0.62068  1.23346  2.15919]
largest coefficient difference after 30 iterations: 4.08e-01


Thirty passes over the data, and it is still a visible distance away in the largest coefficient
from where one pass had already arrived exactly. It is converging, and enough further iterations would close
the gap to any tolerance asked for — but every one of them is another full scan of the training
set, and the closed form was finished after the first.

That is not an argument against gradient descent. It is the statement of when the closed form
wins, which is whenever it is available at all — and the next section is about how often that is.

---

## 5. What one pass costs, and where it stops being one pass

The closed form reads the data once. Everything it pays after that is a function of $d$ and of
nothing else:

* forming the Gramian costs $O(nd^2)$ arithmetic, distributed;
* **shipping it to the driver costs $d^2$ doubles**, and that is the number that decides the
  method's fate;
* solving the system costs $O(d^3)$ on one machine.

Spark's documentation recommends the normal equations only up to about **4096 features**, and
switches to L-BFGS above that. The recommendation is a memory argument, not an algorithmic one,
and the arithmetic behind it is worth doing rather than quoting.

In [9]:
import pandas as pd

def human(nbytes):
    # Decimal units, the convention the chapter's 134 MB and 20 GB figures use.
    for unit, size in (("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if nbytes >= size:
            return f"{nbytes / size:,.1f} {unit}"
    return f"{nbytes:,.0f} B"


rowsize = [{"features d": dd,
            "entries in X^T X": dd * dd,
            "bytes on the driver": dd * dd * 8,
            "that is": human(dd * dd * 8)}
           for dd in (10, 100, 1000, 4096, 20000, 50000)]
print(pd.DataFrame(rowsize).to_string(index=False))
print("\nSpark's recommended limit is 4096 features; above it the estimators use L-BFGS.")

 features d  entries in X^T X  bytes on the driver  that is
         10               100                  800    800 B
        100             10000                80000  80.0 KB
       1000           1000000              8000000   8.0 MB
       4096          16777216            134217728 134.2 MB
      20000         400000000           3200000000   3.2 GB
      50000        2500000000          20000000000  20.0 GB

Spark's recommended limit is 4096 features; above it the estimators use L-BFGS.


Nothing about the *data* changed across that table. A hundred billion rows would produce exactly
the same numbers, because $X^T X$ never grows with $n$. What breaks is the assumption that the
driver can hold a $d \times d$ matrix — 134 MB is comfortable, 20 GB is not — and once that
assumption goes, so does the whole approach, since the solve happens in one process.

The same ceiling is inherited by anything built on the weighted-least-squares solver.
`GeneralizedLinearRegression` solves one weighted least-squares problem per iteration, so it
carries the 4096-feature limit too, even though it is itself an iterative method.

The measurement below is the other half: the solve is $O(d^3)$, on the driver, and the growth is
visible well below the memory limit.

In [10]:
import time

REPEATS = 5

timings = []
for dd in (400, 800, 1600, 3200):
    A = np.random.rand(dd, dd)
    A = A.T @ A + dd * np.eye(dd)          # symmetric positive definite, like a Gramian
    b = np.random.rand(dd)
    np.linalg.solve(A, b)                  # once to warm the library up, not timed
    t0 = time.perf_counter()
    for _ in range(REPEATS):
        np.linalg.solve(A, b)
    secs = (time.perf_counter() - t0) / REPEATS
    timings.append({"d": dd, "solve seconds": round(secs, 5)})

t = pd.DataFrame(timings)
t["seconds / d^3"] = (t["solve seconds"] / t["d"] ** 3).map(lambda v: f"{v:.2e}")
t["time x, when d x2"] = [""] + [f"{a / b:.1f}" for a, b in
                                 zip(t["solve seconds"][1:], t["solve seconds"][:-1])]
print(t.to_string(index=False))
print("\nTimings are machine-dependent, so read the last two columns rather than the seconds.")
print("Doubling d should multiply the time by about 8, and the constant in the middle column")
print("should stop drifting once d is large enough for the solve to dominate the overhead.")

   d  solve seconds seconds / d^3 time x, when d x2
 400        0.00043      6.72e-12                  
 800        0.00203      3.96e-12               4.7
1600        0.01344      3.28e-12               6.6
3200        0.13834      4.22e-12              10.3

Timings are machine-dependent, so read the last two columns rather than the seconds.
Doubling d should multiply the time by about 8, and the constant in the middle column
should stop drifting once d is large enough for the solve to dominate the overhead.


---

## Conclusion

The exact solution to a linear regression is one distributed pass and one local solve. The pass
produces two summaries — a $d \times d$ matrix and a length-$d$ vector — and neither of them grows
with the number of training examples, which is why the method scales in $n$ without effort.

It does not scale in $d$. The matrix that has to reach the driver holds $d^2$ doubles and the
solve costs $O(d^3)$, so the approach runs into a wall that has nothing to do with how much data
there is. Spark places that wall at about 4096 features and reaches for L-BFGS beyond it.

The two routes are therefore not competitors. Below the wall the closed form is the better choice
almost always: no learning rate to tune, no convergence to monitor, one pass instead of sixty, and
the exact optimum rather than an approach to it. Above the wall it is not a choice at all.